<a href="https://colab.research.google.com/github/I7411/TH_BIGDATA/blob/main/20_huynhthanhminhtam_LAB2_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1.1. Giới thiệu bài toán và xác định thuộc tính

* **Khóa dòng đơn hàng:** `line_id` (định danh duy nhất từng mặt hàng trong đơn).
* **Khóa đơn hàng:** `order_id` (một đơn hàng có thể gồm nhiều dòng `line_id`).
* **Các trường tài chính:** `quantity`, `unit_price_usd`, `discount_rate`, `gross_sales_usd`, `discount_usd`, `net_sales_usd` (kèm `payment_status = "Paid"` khi lọc doanh thu đã thanh toán).
* **Trường bán cấu trúc (JSON):** `session_json` (chứa object `device` gồm `type`, `os` và mảng `touchpoints[]`).
* **Các trường bối cảnh (không cần cho KPI chính):** `campaign_label`, `ui_theme`, `support_queue`.

#### Trả lời Câu hỏi thảo luận (1.1)

1. **Vì sao số dòng đơn hàng không nhất thiết bằng số đơn hàng?**
   * **Trả lời:** Vì dữ liệu được lưu ở mức chi tiết từng mặt hàng (*line-item granularity*): một đơn hàng (`order_id`) có thể mua nhiều sản phẩm (`sku`) khác nhau, mỗi sản phẩm tạo thành một bản ghi `line_id` riêng nên tổng số dòng lớn hơn tổng số đơn hàng.

2. **Điều gì có thể xảy ra nếu tính tổng doanh thu sau khi `explode` mảng `touchpoints` mà không xử lý trùng lặp?**
   * **Trả lời:** Nếu một dòng đơn hàng có $k$ điểm chạm trong mảng `touchpoints`, lệnh `explode` sẽ nhân bản dòng đó thành $k$ hàng giống nhau về giá trị tiền. Khi cộng trực tiếp `net_sales_usd` trên bảng đã `explode`, doanh thu của dòng đơn đó sẽ bị cộng lặp $k$ lần (*double/multiple counting*), làm sai lệch nghiêm trọng KPI doanh thu.

---

### 1.2. Khởi tạo môi trường Google Colab (Notebook mới, không dùng GPU)

**Ô lệnh 1 – Cài thư viện:**

In [1]:
!pip -q install pyspark psutil

Ô lệnh 2 – Khởi tạo SparkSession và kiểm tra cấu hình máy ảo:

In [2]:
import os, shutil, time
from pathlib import Path
import psutil
from pyspark.sql import SparkSession, functions as F, types as T

spark = (
    SparkSession.builder.master("local[2]")
    .appName("Lab02_Distributed_ETL")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark:", spark.version)
print("CPU:", os.cpu_count())
print("RAM GiB:", round(psutil.virtual_memory().total / 2**30, 2))
print("Master:", spark.sparkContext.master)

Spark: 4.0.5
CPU: 2
RAM GiB: 12.67
Master: local[2]


Khi đổi local[2] thành local[4], số máy tính vật lý có thay đổi không?
Không. Spark vẫn chạy trên một máy ảo Colab duy nhất; tham số [2] hay [4] chỉ thay đổi số luồng xử lý song song (worker threads) bên trong JVM của máy ảo đó.

Vì sao phải cố định phiên Spark và ghi lại cấu hình khi so sánh thời gian?
Vì thời gian thực thi chịu ảnh hưởng trực tiếp bởi số luồng CPU, dung lượng RAM, số partition shuffle và cơ chế cache trong cùng một phiên chạy. Cố định cấu hình giúp đảm bảo điều kiện thực nghiệm đồng nhất và có thể đối chiếu công bằng giữa các bước đo.

Ô lệnh 3 – Tạo cấu trúc Data Lake và nạp dữ liệu vào vùng Raw:   


In [5]:
import logging
import shutil
from pathlib import Path
from google.colab import auth, drive
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Ẩn cảnh báo thư viện httplib2
logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)

ROOT = Path("/content/lab02/data_lake")
RAW = ROOT / "raw"
CLEANED = ROOT / "cleaned"
CURATED = ROOT / "curated"

for folder in (RAW, CLEANED, CURATED):
    folder.mkdir(parents=True, exist_ok=True)

FILENAME = "Lab02_Bai1_US_Commerce_Orders.csv"
DATA_PATH = RAW / FILENAME

# Xóa tệp rỗng (0 byte) nếu tồn tại từ lần chạy lỗi trước
if DATA_PATH.is_file() and DATA_PATH.stat().st_size == 0:
    DATA_PATH.unlink()

if not DATA_PATH.is_file():
    # Cách 1: Kiểm tra nếu đã có tệp .csv thật trong Google Drive/BigData_Lab02
    drive.mount("/content/drive")
    source = Path("/content/drive/MyDrive/BigData_Lab02") / FILENAME

    if source.is_file():
        shutil.copy2(source, DATA_PATH)
    else:
        # Cách 2: Lọc đúng tệp .csv gốc (bao gồm tệp được chia sẻ) và tải trực tiếp
        auth.authenticate_user()
        drive_service = build("drive", "v3")
        res = drive_service.files().list(
            q=f"name = '{FILENAME}' and mimeType = 'text/csv' and trashed = false",
            fields="files(id, name, mimeType, size)",
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
            pageSize=5,
        ).execute()
        files = res.get("files", [])
        assert files, f"Không tìm thấy tệp gốc {FILENAME} trên Google Drive!"

        target = files[0]
        request = drive_service.files().get_media(fileId=target["id"])

        with open(DATA_PATH, "wb") as f:
            downloader = MediaIoBaseDownload(f, request)
            done = False
            while not done:
                _, done = downloader.next_chunk()

assert DATA_PATH.is_file() and DATA_PATH.stat().st_size > 0, f"Missing or empty file: {DATA_PATH}"
print("Raw CSV (MiB):", round(DATA_PATH.stat().st_size / (2**20), 2))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Raw CSV (MiB): 61.4


Ô lệnh 4 – Hiển thị tệp đã có trong vùng Raw:   

In [6]:
for f in sorted(RAW.iterdir()):
    print(f.name, round(f.stat().st_size / (2**20), 2), "MiB")

Lab02_Bai1_US_Commerce_Orders.csv 61.4 MiB


Vì sao không nên ghi đè CSV tại tầng Raw?
Tầng Raw đóng vai trò là bản sao lưu nguyên gốc (immutable source of truth). Giữ nguyên vẹn dữ liệu gốc giúp có thể chạy lại pipeline từ đầu hoặc đối soát lỗi bất cứ lúc nào nếu quy tắc làm sạch ở các tầng sau thay đổi

Nếu 40 sinh viên cùng đọc từ một tệp Drive dùng chung thì phép đo hiệu năng có thể bị ảnh hưởng như thế nào?
Việc truy cập đồng thời vào một tệp mạng dùng chung sẽ gây nghẽn băng thông mạng, giới hạn hạn ngạch truy xuất API của Google Drive (rate limiting) và tạo độ trễ I/O biến động mạnh, khiến thời gian đo không phản ánh đúng năng lực xử lý của Spark.

1.4. Đọc CSV bằng schema tường minh
Ô lệnh 5 – Khai báo schema nguồn (20 cột):

In [7]:
RAW_SCHEMA = T.StructType([
    T.StructField("line_id", T.LongType()),
    T.StructField("order_id", T.LongType()),
    T.StructField("order_date", T.StringType()),
    T.StructField("customer_id", T.LongType()),
    T.StructField("city", T.StringType()),
    T.StructField("state", T.StringType()),
    T.StructField("channel", T.StringType()),
    T.StructField("category", T.StringType()),
    T.StructField("sku", T.StringType()),
    T.StructField("quantity", T.IntegerType()),
    T.StructField("unit_price_usd", T.DoubleType()),
    T.StructField("discount_rate", T.DoubleType()),
    T.StructField("gross_sales_usd", T.DoubleType()),
    T.StructField("discount_usd", T.DoubleType()),
    T.StructField("net_sales_usd", T.DoubleType()),
    T.StructField("payment_status", T.StringType()),
    T.StructField("session_json", T.StringType()),
    T.StructField("campaign_label", T.StringType()),
    T.StructField("ui_theme", T.StringType()),
    T.StructField("support_queue", T.StringType()),
])

Ô lệnh 6 – Đọc CSV và xác nhận số dòng, khóa chính, schema:   

In [8]:
raw = (
    spark.read.schema(RAW_SCHEMA)
    .option("header", "true")
    .option("escape", '"')
    .option("mode", "FAILFAST")
    .csv(str(DATA_PATH))
)

print("Rows:", raw.count())
print("Columns:", len(raw.columns))
print("Distinct line IDs:", raw.select("line_id").distinct().count())
raw.printSchema()
raw.select("line_id", "order_id", "category", "session_json").show(3, truncate=90)

Rows: 300000
Columns: 20
Distinct line IDs: 300000
root
 |-- line_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sku: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price_usd: double (nullable = true)
 |-- discount_rate: double (nullable = true)
 |-- gross_sales_usd: double (nullable = true)
 |-- discount_usd: double (nullable = true)
 |-- net_sales_usd: double (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- session_json: string (nullable = true)
 |-- campaign_label: string (nullable = true)
 |-- ui_theme: string (nullable = true)
 |-- support_queue: string (nullable = true)

+---------+--------+-----------+-----------------------------------------------------------------------------

Vì sao cần kiểm tra cả count() và số khóa line_id phân biệt?
Để xác minh tính duy nhất của khóa chính: nếu count() bằng distinct().count() (đều bằng 300.000) thì tập dữ liệu không bị trùng lặp bản ghi dòng hàng; ngược lại sẽ phát hiện ngay dữ liệu bị nhân bản ngay từ nguồn.

Những rủi ro nào có thể xảy ra nếu đọc discount_rate thành kiểu chuỗi (StringType)?
Không thể thực hiện trực tiếp các phép nhân/trừ số học để kiểm tra công thức tính chiết khấu (G * r), dễ gây lỗi khi so sánh điều kiện số hoặc tự động ép kiểu ngầm định làm phát sinh giá trị null ngoài ý muốn.

1.5. Phân tích JSON lồng nhau và mảng bằng Spark
Ô lệnh 7 – Định nghĩa schema JSON và trích xuất trường lồng nhau:

In [9]:
SESSION_SCHEMA = T.StructType([
    T.StructField("device", T.StructType([
        T.StructField("type", T.StringType()),
        T.StructField("os", T.StringType()),
    ])),
    T.StructField("touchpoints", T.ArrayType(T.StringType())),
])

parsed = raw.withColumn(
    "session", F.from_json(F.col("session_json"), SESSION_SCHEMA)
)

parsed.select(
    "line_id", "session.device.type", "session.device.os", "session.touchpoints"
).show(5, truncate=False)
print("Invalid JSON rows:", parsed.filter(F.col("session").isNull()).count())

+---------+-------+-------+---------------+
|line_id  |type   |os     |touchpoints    |
+---------+-------+-------+---------------+
|310000001|Desktop|Windows|[Search, Email]|
|310000002|Desktop|Windows|[Search, Email]|
|310000003|Desktop|Windows|[Search, Email]|
|310000004|Mobile |iOS    |[Direct]       |
|310000005|Mobile |iOS    |[Direct]       |
+---------+-------+-------+---------------+
only showing top 5 rows
Invalid JSON rows: 0


Ô lệnh 8 – Sử dụng explode_outer và so sánh số dòng trước/sau tách mảng:

In [10]:
touchpoints = parsed.select(
    "line_id",
    "category",
    F.explode_outer(F.col("session.touchpoints")).alias("touchpoint"),
)

print("Before explode:", parsed.count())
print("After explode:", touchpoints.count())
touchpoints.groupBy("touchpoint").count().orderBy("touchpoint").show()

Before explode: 300000
After explode: 350001
+----------+------+
|touchpoint| count|
+----------+------+
|    Direct|150000|
|     Email| 50001|
|   InStore| 99999|
|    Search| 50001|
+----------+------+



Vì sao số dòng sau explode có thể lớn hơn 300.000?
Vì nhiều dòng đơn hàng có mảng touchpoints chứa từ 2 phần tử trở lên (ví dụ ["Search", "Email"]), nên explode_outer tách mỗi phần tử mảng thành một dòng độc lập, làm tổng số dòng tăng lên.

Nếu một đơn hàng gắn hai touchpoints, điều gì xảy ra khi cộng doanh thu theo bảng đã explode?
Giá trị doanh thu của đơn hàng đó sẽ bị lặp lại ở cả hai dòng và bị cộng gấp đôi vào tổng doanh thu (fan-out trap).

Điểm khác nhau giữa struct và array trong Spark là gì?
struct là tập hợp các trường có tên và kiểu dữ liệu xác định, truy xuất bằng dấu chấm (như session.device.type) và giữ nguyên số dòng; còn array là danh sách tuần tự các phần tử cùng kiểu có độ dài thay đổi theo từng dòng, cần dùng explode/explode_outer để trải phẳng thành các hàng riêng.

1.6. Tạo và đọc dữ liệu JSON Lines
Ô lệnh 9 – Ghi JSON Lines giữ nguyên cấu trúc lồng nhau và đọc lại:

In [11]:
JSON_DIR = ROOT / "json_events"
export_json = parsed.select("line_id", "order_id", "order_date", "category", "session")

(
    export_json.coalesce(4)
    .write.mode("overwrite")
    .json(str(JSON_DIR))
)

json_df = spark.read.json(str(JSON_DIR))
print("JSON rows:", json_df.count())
json_df.printSchema()
json_df.select("line_id", "session.device.type").show(5)

JSON rows: 300000
root
 |-- category: string (nullable = true)
 |-- line_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_id: long (nullable = true)
 |-- session: struct (nullable = true)
 |    |-- device: struct (nullable = true)
 |    |    |-- os: string (nullable = true)
 |    |    |-- type: string (nullable = true)
 |    |-- touchpoints: array (nullable = true)
 |    |    |-- element: string (containsNull = true)

+---------+-------+
|  line_id|   type|
+---------+-------+
|310000001|Desktop|
|310000002|Desktop|
|310000003|Desktop|
|310000004| Mobile|
|310000005| Mobile|
+---------+-------+
only showing top 5 rows


Vì sao thư mục JSON được ghi ra nhiều tệp part-*.json thay vì một tệp?
Vì Spark xử lý và ghi dữ liệu song song theo từng partition. Ở lệnh trên gọi coalesce(4) nên 4 partition ghi đồng thời ra 4 tệp part-00000... đến part-00003..., giúp tránh nút thắt cổ chai khi dồn toàn bộ dữ liệu về một luồng ghi.

Đọc JSON Lines có bắt buộc phải flatten tất cả trường lồng nhau không?
Không bắt buộc. Spark hỗ trợ đầy đủ kiểu dữ liệu lồng nhau (StructType, ArrayType) nên có thể lưu giữ nguyên cấu trúc phân cấp và truy vấn trực tiếp qua đường dẫn lồng nhau (ví dụ session.device.type) khi cần.

1.7. Xây dựng kiểm tra chất lượng dữ liệu (Data Quality Gate)
Ô lệnh 10 – Tạo cờ kiểm tra valid_record:

In [12]:
validated = (
    parsed.withColumn("order_dt", F.to_date("order_date", "yyyy-MM-dd"))
    .withColumn("expected_gross", F.round(F.col("quantity") * F.col("unit_price_usd"), 2))
    .withColumn("expected_discount", F.round(F.col("expected_gross") * F.col("discount_rate"), 2))
    .withColumn("expected_net", F.round(F.col("expected_gross") - F.col("expected_discount"), 2))
)

condition = (
    F.col("line_id").isNotNull()
    & F.col("order_dt").isNotNull()
    & F.col("session").isNotNull()
    & (F.col("quantity") > 0)
    & (F.abs(F.col("gross_sales_usd") - F.col("expected_gross")) <= 0.011)
    & (F.abs(F.col("discount_usd") - F.col("expected_discount")) <= 0.011)
    & (F.abs(F.col("net_sales_usd") - F.col("expected_net")) <= 0.011)
)

validated = validated.withColumn("valid_record", condition)
validated.groupBy("valid_record").count().show()

+------------+------+
|valid_record| count|
+------------+------+
|        true|300000|
+------------+------+



Ô lệnh 11 – Tách vùng lỗi và đếm số dòng hợp lệ:

In [13]:
invalid = validated.filter(~F.col("valid_record"))
print("Invalid rows:", invalid.count())
print("Valid rows:", validated.filter("valid_record").count())
print("Null dates:", validated.filter(F.col("order_dt").isNull()).count())

Invalid rows: 0
Valid rows: 300000
Null dates: 0


Vì sao không tự ý thay gross_sales_usd sai lệch bằng 0?
Thay thế bằng 0 sẽ làm sai lệch bản chất nghiệp vụ (biến một giao dịch có phát sinh doanh thu thành giao dịch bằng 0) và che giấu lỗi hệ thống nguồn thay vì sửa đúng nguyên nhân.

Vì sao lỗi dữ liệu cần được ghi nhận trước khi loại bỏ?
Để phục vụ truy vết (data lineage / auditing), xác định nguyên nhân phát sinh lỗi từ nguồn nhập liệu và báo cáo tỷ lệ dữ liệu bị loại bỏ nhằm đảm bảo tính minh bạch của pipeline.

Tại sao dùng sai số 0.011 khi đối chiếu giá trị USD đã làm tròn?
Do phép biểu diễn số thực dấu phẩy động nhị phân (DoubleType) và việc làm tròn từng bước đến 2 chữ số thập phân (1 cent = 0.01) có thể tạo ra chênh lệch làm tròn tối đa 0.01 USD; dùng ngưỡng 0.011 tránh loại nhầm các bản ghi đúng chỉ vì sai số biểu diễn số thực nhỏ hơn 1 cent.

1.8. Tạo vùng Cleaned và Curated trong Data Lake

Ô lệnh 12 – Tạo tầng Cleaned (Silver):

In [14]:
silver = (
    validated.filter(F.col("valid_record"))
    .drop("expected_gross", "expected_discount", "expected_net", "valid_record")
    .dropDuplicates(["line_id"])
    .withColumn("year", F.year("order_dt"))
    .withColumn("month", F.month("order_dt"))
)

SILVER_DIR = CLEANED / "orders_parquet"
silver.write.mode("overwrite").parquet(str(SILVER_DIR))
print("Cleaned rows:", spark.read.parquet(str(SILVER_DIR)).count())

Cleaned rows: 300000


Ô lệnh 13 – Tạo tầng Curated (Gold):

In [15]:
gold = (
    silver.filter(F.col("payment_status") == "Paid")
    .groupBy("year", "month", "category")
    .agg(
        F.count("*").alias("paid_lines"),
        F.sum("quantity").alias("total_units"),
        F.round(F.sum("net_sales_usd"), 2).alias("sales_usd"),
    )
)

GOLD_DIR = CURATED / "monthly_category_sales"
(
    gold.write.mode("overwrite")
    .partitionBy("year", "month")
    .parquet(str(GOLD_DIR))
)

print("Curated groups:", spark.read.parquet(str(GOLD_DIR)).count())
gold.orderBy("year", "month", "category").show(10)

Curated groups: 72
+----+-----+-----------+----------+-----------+---------+
|year|month|   category|paid_lines|total_units|sales_usd|
+----+-----+-----------+----------+-----------+---------+
|2025|    1|     Beauty|      4025|       9390| 184457.5|
|2025|    1|      Books|      4023|       9385|210388.95|
|2025|    1|Electronics|      4024|       9391| 932781.6|
|2025|    1|       Home|      4024|       9391| 641085.8|
|2025|    1|     Office|      4022|       9386|231785.31|
|2025|    1|     Sports|      4023|       9386| 501720.8|
|2025|    2|     Beauty|      3634|       8479| 166557.5|
|2025|    2|      Books|      3633|       8477|190066.45|
|2025|    2|Electronics|      3635|       8482| 842538.6|
|2025|    2|       Home|      3634|       8479| 578699.8|
+----+-----+-----------+----------+-----------+---------+
only showing top 10 rows


Đâu là sự khác nhau giữa tầng Cleaned và Curated?Cleaned (Silver):
1.   Lưu dữ liệu chi tiết từng dòng đơn hàng đã được chuẩn hóa kiểu dữ liệu, loại trùng khóa chính và vượt qua kiểm tra chất lượng.   
2.   Curated (Gold): Lưu dữ liệu đã được lọc theo quy tắc nghiệp vụ (Paid) và tổng hợp sẵn theo các chiều phân tích (year, month, category) để phục vụ trực tiếp báo cáo và dashboard.

Vì sao phải loại payment_status = Pending khi tính doanh thu đã thanh toán?
Vì các dòng Pending là giao dịch chưa hoàn tất thanh toán (chưa thực thu tiền), nếu gộp chung sẽ làm thổi phồng doanh thu thực tế của doanh nghiệp.


1.9. Ghi Parquet có phân vùng và quan sát Partition Pruning
Ô lệnh 14 – Xuất Parquet có partitionBy("year", "month"):

In [16]:
PART_DIR = CLEANED / "orders_partitioned"
(
    silver.repartition(4, "year", "month")
    .write.mode("overwrite")
    .partitionBy("year", "month")
    .parquet(str(PART_DIR))
)

print("Partition folders:")
for x in sorted(PART_DIR.glob("year=*/month=*"))[:8]:
    print(x.relative_to(PART_DIR))

Partition folders:
year=2025/month=1
year=2025/month=10
year=2025/month=11
year=2025/month=12
year=2025/month=2
year=2025/month=3
year=2025/month=4
year=2025/month=5


Ô lệnh 15 – Truy vấn tháng 7/2025 và kiểm tra PartitionFilters trong Physical Plan:

In [17]:
partitioned = spark.read.parquet(str(PART_DIR))
july = partitioned.filter((F.col("year") == 2025) & (F.col("month") == 7))
print("July rows:", july.count())

july_agg = (
    july.select("category", "net_sales_usd")
    .groupBy("category")
    .agg(F.round(F.sum("net_sales_usd"), 2).alias("sales_usd"))
)
july_agg.show()
july_agg.explain(mode="formatted")

July rows: 25479
+-----------+---------+
|   category|sales_usd|
+-----------+---------+
|      Books| 222017.2|
|       Home| 676527.0|
|     Beauty| 194632.5|
|     Office|244693.95|
|Electronics| 984686.4|
|     Sports| 529571.2|
+-----------+---------+

== Physical Plan ==
AdaptiveSparkPlan (6)
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
         +- Project (2)
            +- Scan parquet  (1)


(1) Scan parquet 
Output [4]: [category#1204, net_sales_usd#1211, year#1219, month#1220]
Batched: true
Location: InMemoryFileIndex [file:/content/lab02/data_lake/cleaned/orders_partitioned]
PartitionFilters: [isnotnull(year#1219), isnotnull(month#1220), (year#1219 = 2025), (month#1220 = 7)]
ReadSchema: struct<category:string,net_sales_usd:double>

(2) Project
Output [2]: [category#1204, net_sales_usd#1211]
Input [4]: [category#1204, net_sales_usd#1211, year#1219, month#1220]

(3) HashAggregate
Input [2]: [category#1204, net_sales_usd#1211]
Keys [1]: [category#1204]
Fu

Khác nhau giữa partitionBy(year, month) và repartition(4) là gì?
partitionBy("year", "month") là phân vùng lưu trữ trên đĩa (storage partitioning), tổ chức dữ liệu thành các thư mục con year=.../month=.... Còn repartition(4) là phân vùng tác vụ trong bộ nhớ (in-memory partitioning), chia dữ liệu thành 4 phần để xử lý song song và quyết định số tệp part-* được ghi ra trong mỗi thư mục.   Vì sao lọc trực tiếp year/month giúp khai thác partition pruning?

Vì giá trị của year và month được mã hóa ngay trên tên thư mục lưu trữ (year=2025/month=7). Khi câu truy vấn lọc trực tiếp trên hai cột này, bộ tối ưu Catalyst nhận diện được thư mục khớp điều kiện từ metadata mà không cần mở hay đọc dữ liệu bên trong các thư mục tháng khác.   Chuyện gì xảy ra khi partition dữ liệu quá nhỏ tạo nhiều file?

Sẽ gây ra hiện tượng nhiều tệp nhỏ (small files problem): làm phình to chi phí quản lý siêu dữ liệu (metadata), tăng thời gian mở/đóng từng tệp trên hệ thống lưu trữ và làm chậm đáng kể tốc độ đọc truy vấn về sau.   

1.10. So sánh lưu trữ CSV và Parquet trên cùng dữ liệu
Ô lệnh 16 – Ghi cùng tập dữ liệu con ra CSV và Parquet (4 part-file):

In [18]:
COLS = ["order_id", "category", "year", "month", "quantity", "net_sales_usd"]
subset = silver.select(*COLS)

CSV_OUT = ROOT / "compare" / "csv"
PARQUET_OUT = ROOT / "compare" / "parquet"

(
    subset.coalesce(4)
    .write.mode("overwrite")
    .option("header", "true")
    .csv(str(CSV_OUT))
)
(
    subset.coalesce(4)
    .write.mode("overwrite")
    .parquet(str(PARQUET_OUT))
)

def data_bytes(folder):
    return sum(
        p.stat().st_size
        for p in folder.rglob("*")
        if p.is_file() and not p.name.startswith(".") and not p.name.startswith("_")
    )

print("CSV MiB:", round(data_bytes(CSV_OUT) / 2**20, 2))
print("Parquet MiB:", round(data_bytes(PARQUET_OUT) / 2**20, 2))

CSV MiB: 8.64
Parquet MiB: 1.69


Ô lệnh 17 – Benchmark cùng phép tổng hợp trên CSV và Parquet (đo 3 lần lấy trung vị):

In [19]:
import statistics

COMPARE_SCHEMA = T.StructType([
    T.StructField("order_id", T.LongType()),
    T.StructField("category", T.StringType()),
    T.StructField("year", T.IntegerType()),
    T.StructField("month", T.IntegerType()),
    T.StructField("quantity", T.IntegerType()),
    T.StructField("net_sales_usd", T.DoubleType()),
])

def compare_job(fmt):
    start = time.perf_counter()
    if fmt == "CSV":
        frame = (
            spark.read.schema(COMPARE_SCHEMA)
            .option("header", "true")
            .csv(str(CSV_OUT))
        )
    else:
        frame = spark.read.parquet(str(PARQUET_OUT))

    rows = (
        frame.filter(F.col("month") == 7)
        .groupBy("category")
        .agg(F.sum("net_sales_usd").alias("sales"))
        .collect()
    )
    return time.perf_counter() - start, len(rows)

for fmt in ["CSV", "Parquet"]:
    times = []
    n_groups = 0
    for _ in range(3):
        sec, n_groups = compare_job(fmt)
        times.append(sec)
    print(f"{fmt}: median = {round(statistics.median(times), 4)} s | runs = {[round(x, 4) for x in times]} | groups = {n_groups}")

CSV: median = 0.8424 s | runs = [1.3299, 0.7761, 0.8424] | groups = 6
Parquet: median = 0.4324 s | runs = [0.5229, 0.4324, 0.3749] | groups = 6


Tại sao hai định dạng phải được so sánh trên cùng cột và số dòng?
Để cô lập duy nhất biến số là định dạng lưu trữ. Nếu khác số cột hoặc số dòng, sự chênh lệch về dung lượng đĩa và thời gian đọc sẽ bị chi phối bởi khối lượng dữ liệu khác nhau chứ không phản ánh đúng ưu thế nén và truy xuất của Parquet so với CSV.   Truy vấn chỉ đọc category và net_sales_usd có thể được lợi từ Parquet như thế nào?
Nhờ cấu trúc lưu trữ hướng cột (columnar storage), Parquet hỗ trợ cắt tỉa cột (column pruning): Spark chỉ đọc đúng hai khối cột category, net_sales_usd (và cột lọc month) từ đĩa mà bỏ qua hoàn toàn các cột còn lại, đồng thời tận dụng thống kê min/max từng khối (predicate pushdown) và giải nén dữ liệu nhanh hơn so với phân tích chuỗi văn bản CSV.   Vì sao một lần chạy có thể không chứng minh được định dạng nào nhanh hơn?
Vì thời gian đọc ở một lần chạy đơn lẻ trên Colab rất dễ bị nhiễu bởi cơ chế bộ nhớ đệm trang (OS page cache) của hệ điều hành (lần đọc sau lấy dữ liệu từ RAM thay vì ổ đĩa) và sự biến động tài nguyên CPU dùng chung giữa các phiên.

1.11. Kiểm tra pipeline đầu cuối và hoàn thiện kết quả
Ô lệnh 18 – Kiểm tra số dòng các tầng và đối soát KPI doanh thu Paid:

In [20]:
raw_rows = raw.count()
clean_rows = spark.read.parquet(str(SILVER_DIR)).count()
curated_rows = spark.read.parquet(str(GOLD_DIR)).count()

paid_total = (
    silver.filter(F.col("payment_status") == "Paid")
    .agg(F.sum("net_sales_usd"))
    .first()[0]
)
gold_total = gold.agg(F.sum("sales_usd")).first()[0]

print("Raw:", raw_rows, "| Cleaned:", clean_rows, "| Curated:", curated_rows)
print("Paid total (Cleaned):", round(paid_total, 2))
print("Curated total (Gold):", round(gold_total, 2))

assert abs(paid_total - gold_total) < 1.0
assert raw_rows == clean_rows == 300_000
print("\n[HOÀN TẤT]: Pipeline ETL đầu cuối đạt chuẩn toàn vẹn dữ liệu!")

Raw: 300000 | Cleaned: 300000 | Curated: 72
Paid total (Cleaned): 31810897.94
Curated total (Gold): 31810897.94

[HOÀN TẤT]: Pipeline ETL đầu cuối đạt chuẩn toàn vẹn dữ liệu!


### Câu hỏi thảo luận về Pipeline và Data Lake

1. **Trong pipeline này, hai bước nào thể hiện rõ xử lý Big Data thay vì phân tích bảng thông thường?**

   * **Tổ chức Data Lake đa tầng (`Raw` – `Cleaned` – `Curated`) kết hợp lưu trữ phân tán dạng cột (`Parquet` phân vùng thư mục `year/month` và ghi đa tệp `part-*`):** Cho phép mở rộng lưu trữ hàng terabyte và khai thác *partition pruning* / *column pruning* thay vì lưu một tệp bảng đơn lẻ.

   * **Xử lý dữ liệu bán cấu trúc lồng nhau phân tán (`from_json`, `explode_outer`) kết hợp *Data Quality Gate* trước khi tổng hợp:** Tự động kiểm tra toàn vẹn hàng trăm nghìn bản ghi theo *schema* tường minh trên các *partition* song song.

2. **Nếu lần chạy sau có thêm dữ liệu năm 2026, cần kiểm tra những gì trước khi cập nhật `Curated`?**

   * **Trả lời:** Cần kiểm tra:
     * **(i)** Cấu trúc 20 cột nguồn và *schema* của `session_json` có giữ nguyên định dạng hay không;
     * **(ii)** Khoảng giá trị ngày `order_date` thuộc năm 2026 và không bị trùng `line_id` với dữ liệu lịch sử năm 2025;
     * **(iii)** Tỷ lệ bản ghi vi phạm quy tắc tính tiền (`gross`, `discount`, `net`) qua *Data Quality Gate*;
     * **(iv)** Cơ chế ghi đè theo phân vùng (`partitionOverwriteMode="dynamic"`) để không làm mất dữ liệu năm 2025 khi cập nhật tầng `Curated`.

3. **Vì sao cần đối chiếu KPI giữa `Cleaned` và `Curated`?**

   * **Trả lời:** Vì tầng `Curated` có số dòng nhỏ hơn nhiều so với tầng `Cleaned` do đã gom nhóm theo tháng và danh mục. Đối chiếu tổng doanh thu `Paid` giữa hai tầng (`paid_total == gold_total`) là chốt kiểm soát bắt buộc để khẳng định quá trình lọc, nhóm và ghi phân vùng không làm thất thoát hay nhân bản dữ liệu.